In [ ]:
"""This notebook applies quality control filters to the raw UKB cognitive data. It is designed to be applied to the output from the RAP. 
The RAP was used to filter data then export the filtered data. Helper IDs were used such a MP puzzles viewed, which was exported and used as the basis for filtering."""

import pandas as pd

ukb_cog = pd.read_csv('ukb_cog_clean.csv', low_memory=False) #replace with data path
ukb_cog_filtered = ukb_cog.copy()
ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"MP_puzzles shape: {mp_puzzles.shape}")
print(f"UKB_cog shape: {ukb_cog.shape}")
print(f"Overlapping IDs: {len(set(mp_puzzles['eid']).intersection(set(ukb_cog['eid'])))}")

In [ ]:
"""Matrix pattern recognition data, using the data of number of puzzles viewed, filtering for those who viewed less than 4 puzzles, indicating poor effort or understanding of the task"""

mp_puzzles = pd.read_csv('MP_puzzles_viewed.csv', low_memory=False) #replace with data path
mp_columns = ['MP-5', 'MP-2', 'MP-3']
for col in mp_columns:
    if col in ukb_cog_filtered.columns:  # Check if column exists in ukb_cog
        # Get IDs where this MP column has value < 4
        ids_to_filter = mp_puzzles[mp_puzzles[col] < 4]['eid']
        
        # Set corresponding values to empty string for those specific IDs
        mask = ukb_cog_filtered['eid'].isin(ids_to_filter)
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"\nFor {col}:")
        print(f"  - IDs with {col} < 4 in MP_puzzles: {len(ids_to_filter)}")
        print(f"  - Values erased in ukb_cog: {mask.sum()}")
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Original ukb_cog shape: {ukb_cog.shape}")
print(f"- Filtered ukb_cog shape: {ukb_cog_filtered.shape}")
print(f"- IDs that had data erased: {len(set(mp_puzzles['eid']).intersection(set(ukb_cog['eid'])))}")

In [ ]:
"""Broken letter recognition, designed to filter out those who were unable to recognise the first 5 letters presented, which were all unaltered, indicating poor understanding or inability to read"""


bl_columns = ['BL-2', 'BL-3']
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False) #replace with data path

total_erased = 0
all_erased_values = []

for col in bl_columns:
    if col in ukb_cog_filtered.columns:
        ukb_cog_filtered[col] = pd.to_numeric(ukb_cog_filtered[col], errors='coerce')
        
        # Get the values that will be erased
        mask = (ukb_cog_filtered[col] < 6)
        erased_values = ukb_cog_filtered.loc[mask, col].dropna().tolist()
        all_erased_values.extend(erased_values)
        
        values_under_6 = mask.sum()
        non_nan_to_erase = len(erased_values)
        
        print(f"\nFor {col}:")
        print(f"  - Values < 6 found: {values_under_6}")
        print(f"  - Values actually erased: {non_nan_to_erase}")
        print(f"  - Erased values: {erased_values}")
        print(f"  - Value counts: {pd.Series(erased_values).value_counts().sort_index().to_dict()}")
        
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
        
        total_erased += non_nan_to_erase

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total BL values erased across all columns: {total_erased}")
print(f"- All erased values distribution: {pd.Series(all_erased_values).value_counts().sort_index().to_dict()}")

In [ ]:
"""Symbol digit substitution data - here the notebook filteres out repsondents who scored 0 or extraordinarily high (60)."""

sd_data = pd.read_csv('SD_0_60.csv', ) #replace with data path

sd_columns = ['SD-2', 'SD-3', 'SD-4', 'SD-5']
for col in sd_columns:
    if col in ukb_cog_filtered.columns:
        # Get IDs where this SD column has value = 0 or >= 60
        ids_to_filter = sd_data[(sd_data[col] == 0) | (sd_data[col] >= 60)]['eid']
        mask = ukb_cog_filtered['eid'].isin(ids_to_filter)
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"\nFor {col}:")
        print(f"  - IDs with {col} = 0 or >= 60 in SD data: {len(ids_to_filter)}")
        print(f"  - Values erased in ukb_cog_filtered: {mask.sum()}")
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)
print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Additional IDs that had SD data erased: {len(set(sd_data['eid']).intersection(set(ukb_cog_filtered['eid'])))}")

In [ ]:
"""This filters the scores achieved on Trial Making A to exclude any zeros and any extrordinarily high scores. High scores can indicate the participant ignored the timed nature of the test"""

import pandas as pd

tma_columns = ['TMA-2', 'TMA-3', 'TMA-4', 'TMA-5']
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False) #replace with data path

total_erased = 0
all_erased_values = []

for col in tma_columns:
    if col in ukb_cog_filtered.columns:
        ukb_cog_filtered[col] = pd.to_numeric(ukb_cog_filtered[col], errors='coerce')
        
        # Get the values that will be erased
        mask = (ukb_cog_filtered[col] == 0) | (ukb_cog_filtered[col] > 200)
        erased_values = ukb_cog_filtered.loc[mask, col].dropna().tolist()
        all_erased_values.extend(erased_values)
        
        values_to_erase = mask.sum()
        non_nan_to_erase = len(erased_values)
        
        print(f"\nFor {col}:")
        print(f"  - Values = 0 or > 200 found: {values_to_erase}")
        print(f"  - Values actually erased: {non_nan_to_erase}")
        print(f"  - Erased value counts: {pd.Series(erased_values).value_counts().sort_index().to_dict()}")
        
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
        
        total_erased += non_nan_to_erase

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total TMA values erased across all columns: {total_erased}")
print(f"- All erased values distribution: {pd.Series(all_erased_values).value_counts().sort_index().to_dict()}")

In [ ]:
"""As per TMA except with higher threshold for exclusion due to TMB being a naturally more difficult task"""

import pandas as pd

tmb_columns = ['TMB-2', 'TMB-3', 'TMB-4', 'TMB-5']
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False) #replace with data path

total_erased = 0
all_erased_values = []

for col in tmb_columns:
    if col in ukb_cog_filtered.columns:
        ukb_cog_filtered[col] = pd.to_numeric(ukb_cog_filtered[col], errors='coerce')
        
        # Get the values that will be erased
        mask = (ukb_cog_filtered[col] == 0) | (ukb_cog_filtered[col] > 400)
        erased_values = ukb_cog_filtered.loc[mask, col].dropna().tolist()
        all_erased_values.extend(erased_values)
        
        values_to_erase = mask.sum()
        non_nan_to_erase = len(erased_values)
        
        print(f"\nFor {col}:")
        print(f"  - Values = 0 or > 400 found: {values_to_erase}")
        print(f"  - Values actually erased: {non_nan_to_erase}")
        print(f"  - Erased value counts: {pd.Series(erased_values).value_counts().sort_index().to_dict()}")
        
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
        
        total_erased += non_nan_to_erase

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total TMB values erased across all columns: {total_erased}")
print(f"- All erased values distribution: {pd.Series(all_erased_values).value_counts().sort_index().to_dict()}")

In [ ]:
"""This block filters Pairs Matching task with extremely high results which indicate repeated errors and suggests invalid results"""

import pandas as pd
pmat_columns = ['PMat-0', 'PMat-1', 'PMat-2', 'PMat-3', 'PMat-4',]
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False) #replace with data path
for col in pmat_columns:
    if col in ukb_cog_filtered.columns:
        ukb_cog_filtered[col] = pd.to_numeric(ukb_cog_filtered[col], errors='coerce')
        mask = (ukb_cog_filtered[col] > 30)
        values_to_erase = mask.sum()
        ukb_cog_filtered.loc[mask, col] = ''
        print(f"\nFor {col}:")
        print(f"  - Values > 30: {values_to_erase}")
        print(f"  - Values erased: {values_to_erase}")
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)
print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total PMat values erased across all columns")

In [ ]:
"""Fluid Intelligence - this block filters out participants who made 0 correct responses indicating poor understanding of the task or an invalid result"""

import pandas as pd

fi_0 = pd.read_csv('FI_0.csv', low_memory=False) #replace with data path
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False) #replace with data path

print(f"FI_0 shape: {fi_0.shape}")
print(f"UKB_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"Overlapping IDs: {len(set(fi_0['eid']).intersection(set(ukb_cog_filtered['eid'])))}")

fi_columns = ['FI-0', 'FI-1', 'FI-2', 'FI-3', 'FI-4', 'FI-5']
total_erased = 0

for col in fi_columns:
    if col in ukb_cog_filtered.columns and col in fi_0.columns:
        # Find IDs where FI value = 0
        ids_to_filter = fi_0[fi_0[col] == 0]['eid']
        mask = ukb_cog_filtered['eid'].isin(ids_to_filter)
        
        # Count non-empty values before erasing
        non_empty_before = (ukb_cog_filtered.loc[mask, col] != '').sum()
        
        print(f"\nFor {col}:")
        print(f"  - IDs with {col} = 0 in FI_0: {len(ids_to_filter)}")
        print(f"  - Rows matched in ukb_cog_filtered: {mask.sum()}")
        print(f"  - Non-empty values before erasing: {non_empty_before}")
        
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"  - Values actually erased: {non_empty_before}")
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
        
        total_erased += non_empty_before

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total FI values erased across all columns: {total_erased}")

import pandas as pd

fi_0 = pd.read_csv('FI_0.csv', low_memory=False) #replace with data path
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False) #replace with data path

print(f"FI_0 shape: {fi_0.shape}")
print(f"UKB_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"Overlapping IDs: {len(set(fi_0['eid']).intersection(set(ukb_cog_filtered['eid'])))}")

fi_columns = ['FI-4', 'FI-5']
total_erased = 0

for col in fi_columns:
    if col in ukb_cog_filtered.columns and col in fi_0.columns:
        # Find IDs where FI value = 0
        ids_to_filter = fi_0[fi_0[col] == 0]['eid']
        mask = ukb_cog_filtered['eid'].isin(ids_to_filter)
        
        # Count non-empty values before erasing
        non_empty_before = (ukb_cog_filtered.loc[mask, col] != '').sum()
        
        print(f"\nFor {col}:")
        print(f"  - IDs with {col} = 0 in FI_0: {len(ids_to_filter)}")
        print(f"  - Rows matched in ukb_cog_filtered: {mask.sum()}")
        print(f"  - Non-empty values before erasing: {non_empty_before}")
        
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"  - Values actually erased: {non_empty_before}")
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
        
        total_erased += non_empty_before

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total FI values erased across all columns: {total_erased}")

In [ ]:
"""Tower reconstruction task - this block filters those who made 0 correct responses indicating poor effort or understanding, and an invalid result"""

import pandas as pd

tr_0 = pd.read_csv('TR_0.csv', low_memory=False) #replace with data path 
ukb_cog_filtered = pd.read_csv('ukb_cog_filtered.csv', low_memory=False)

print(f"TR_0 shape: {tr_0.shape}")
print(f"UKB_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"Overlapping IDs: {len(set(tr_0['eid']).intersection(set(ukb_cog_filtered['eid'])))}")

tr_columns = ['TR-2', 'TR-3']
total_erased = 0

for col in tr_columns:
    if col in ukb_cog_filtered.columns and col in tr_0.columns:
        # Find IDs where TR value = 0
        ids_to_filter = tr_0[tr_0[col] == 0]['eid']
        mask = ukb_cog_filtered['eid'].isin(ids_to_filter)
        
        # Count non-empty values before erasing
        non_empty_before = (ukb_cog_filtered.loc[mask, col] != '').sum()
        
        print(f"\nFor {col}:")
        print(f"  - IDs with {col} = 0 in TR_0: {len(ids_to_filter)}")
        print(f"  - Rows matched in ukb_cog_filtered: {mask.sum()}")
        print(f"  - Non-empty values before erasing: {non_empty_before}")
        
        ukb_cog_filtered.loc[mask, col] = ''
        
        print(f"  - Values actually erased: {non_empty_before}")
        print(f"  - Remaining non-empty values in {col}: {(ukb_cog_filtered[col] != '').sum()}")
        
        total_erased += non_empty_before

ukb_cog_filtered.to_csv('ukb_cog_filtered.csv', index=False)

print(f"\nSummary:")
print(f"- Updated ukb_cog_filtered shape: {ukb_cog_filtered.shape}")
print(f"- Total TR values erased across all columns: {total_erased}")